# adaLN-Zero — 面试版

**难度：** Medium

**面试要点：** 手写 LayerNorm + 分段条件调制

In [ ]:
import torch

In [ ]:
# ✅ INTERVIEW

def adaln_zero(x, cond, W_ada, b_ada):
    B, N, D = x.shape

    params = cond @ W_ada + b_ada  # (B, 6*D)

    # ---- 手写 chunk（面试可能要求不用 .chunk()）----
    # chunk(6) 等价于按最后一维均匀切片
    # 每段大小 = 6*D / 6 = D
    gamma1 = params[:, 0*D:1*D]
    beta1  = params[:, 1*D:2*D]
    alpha1 = params[:, 2*D:3*D]

    # ---- 手写 LayerNorm ----
    # 逐元素计算，面试时可以更详细地展开
    mean = x.sum(dim=-1, keepdim=True) / D  # 手动求均值
    # var = E[x^2] - E[x]^2（另一种计算方差的方式）
    var = (x ** 2).sum(dim=-1, keepdim=True) / D - mean ** 2
    x_norm = (x - mean) / (var + 1e-6).sqrt()

    # ---- 条件调制 ----
    # α * (γ * x_norm + β)
    out = alpha1.unsqueeze(1) * (gamma1.unsqueeze(1) * x_norm + beta1.unsqueeze(1))
    return out

In [ ]:
torch.manual_seed(0)
B, N, D = 4, 16, 32
C = 16

x    = torch.randn(B, N, D)
cond = torch.randn(B, C)

W_ada = torch.zeros(C, 6 * D)
b_ada = torch.zeros(6 * D)
out_zero = adaln_zero(x, cond, W_ada, b_ada)
print(f"Zero params => max abs output: {out_zero.abs().max().item():.6f}  (expected 0.0)")

In [ ]:
from torch_judge import check
check("adaln_zero")

## 📝 核心思路总结

1. **手动切片替代 chunk**：`params[:, 0*D:1*D]` 等价于 `chunk(6)[0]`
2. **方差公式**：`var = E[x²] - (E[x])²`，另一种计算方式
3. **adaLN-Zero 设计**：门控 α 零初始化 → 初始恒等映射